In [5]:
from pathlib import Path
import cv2
import numpy as np
from mmdet.apis import inference_detector, init_detector

In [25]:
config_path = Path(r"C:\dev\projects\CV_counting_bags\configs\rtmdet_tiny_bag.py")
ckpt_path = Path(r"C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth")
video_path = Path(r"C:\dev\projects\CV_counting_bags\input.mp4")
out_video_path = Path(r"C:\dev\projects\CV_counting_bags\output_video\output.mp4")

In [21]:
model = init_detector(
    str(config_path),
    str(ckpt_path),
    device = "cuda:0"
)

Loads checkpoint by local backend from path: C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth


In [22]:
def draw_detection(frame, result, threshold = 0.36):
    image = frame.copy()

    pred = result.pred_instances
    bboxes = pred.bboxes.detach().cpu().numpy()
    scores = pred.scores.detach().cpu().numpy()

    detections_sum = 0

    for bbox, score in zip(bboxes, scores):
        if score < threshold:
            continue

        detections_sum += 1

        x1, y1, x2, y2 = bbox.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"bag {score:.2f}"

        cv2.putText(image, label, (x1, max(y1 - 7, 20)), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(image, f"Detections: {detections_sum}",
        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    return image

In [26]:
def process_video(viedo_path, out_path, model, threshold = 0.36, max_frames = None):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("failed to open video:", video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"FPS: {fps}")
    print(f"Res: {width}x{height}")
    print(f"Frames: {frames_count}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents = True, exist_ok = True)
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError("failed to create video", out_path)

    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if max_frames is not None and frame_index >= max_frames:
            break

        result = inference_detector(model, frame)
        annotation = draw_detection(frame, result)

        writer.write(annotation)
        frame_index += 1

    cap.release()
    writer.release()
    print("done")

In [27]:
process_video(video_path, out_path = out_video_path, model = model, max_frames = 1500)

FPS: 25.0
Res: 640x360
Frames: 14999
done
